# Notebook 41 — Gradual magnitude pruning with recovery: does the training-time recipe also starve the first layer?

The disparate-impact literature (Hooker et al.; Tran et al.) pruned gradually during training with pruned
weights allowed to recover (Zhu and Gupta, 2017). Notebooks 34-38 used one-shot pruning with hard masks.
This notebook asks whether the input-layer mechanism holds under the gradual recipe.

**Design.** On the five CNN baselines (`M0_paired`, seeds 0-4), sparsity follows the cubic schedule
s_t = s_f + (s_0 - s_f)(1 - t/T)^3 from 0 to 80% over the first 8 fine-tuning epochs, with the mask
recomputed from the *current* weights every 100 steps. Between mask updates, masked weights still receive
gradient updates, so a weight can regrow and be selected back (recovery). After the schedule, 2 more epochs
fine-tune under the final fixed mask. Two allocation rules: per-layer (each Linear/Conv1d pruned to s_t
independently, the default) and global (one threshold at s_t over all prunable weights). Per-layer sparsity,
live conv.0 filters and per-class damage are recorded.

**Gate (stated before running).** The mechanism holds under gradual pruning if per-layer gradual pruning
leaves conv.0 at about 80% sparsity with mean macro-F1 loss > 0.15 and at least four classes materially
affected in >= 3/5 seeds, while global gradual pruning keeps most of conv.0 and loses < 0.05. If per-layer
gradual pruning avoids the collapse (recovery repairs the input layer), the paper scopes its claim to
one-shot pruning. Any outcome is reported. Resumable per (seed, rule). GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'; ARCH = 'cnn1d'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
RULES = ['gradual_perlayer80', 'gradual_global80']
S_FINAL = 0.80; SCHED_EPOCHS = 8; FINAL_EPOCHS = 2; MASK_EVERY = 100; BATCH = 4096; LR = 5e-4

ARCH_KW = {'channels': (64, 128)}; BASE_CELL = 'M0_paired'
print('rules:', RULES, '| schedule to', S_FINAL, 'over', SCHED_EPOCHS, 'epochs, mask every', MASK_EVERY, 'steps, then', FINAL_EPOCHS, 'epochs fixed')

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Shared helpers (as in Notebooks 34-38) plus the gradual pruner.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)


def masks_at(model, s, rule):
    # recompute masks from CURRENT weights at target sparsity s
    params = prunable(model); out = {}
    if rule == 'gradual_perlayer80':
        for mod, name in params:
            w = getattr(mod, name).detach().abs().reshape(-1); k = int(round(s * w.numel()))
            thr = w.kthvalue(k).values if k > 0 else torch.tensor(-1.0, device=w.device)
            out[(mod, name)] = (getattr(mod, name).detach().abs() > thr).float()
    else:
        allw = torch.cat([getattr(mod, name).detach().abs().reshape(-1) for mod, name in params]); k = int(round(s * allw.numel()))
        thr = allw.kthvalue(k).values if k > 0 else torch.tensor(-1.0, device=allw.device)
        for mod, name in params: out[(mod, name)] = (getattr(mod, name).detach().abs() > thr).float()
    return out

def apply_masks(model, masks):
    with torch.no_grad():
        for (mod, name), mk in masks.items(): getattr(mod, name).mul_(mk)

def gradual_prune(anchor, seed, rule, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = copy.deepcopy(anchor).to(DEVICE)
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)
    steps_per_epoch = len(loader); T = SCHED_EPOCHS * steps_per_epoch; step = 0; prev = None
    names = layer_names(model); regrown = {names[mod]: 0 for mod, _ in prunable(model)}
    for ep in range(SCHED_EPOCHS):
        model.train()
        for xb, yb in loader:
            if step % MASK_EVERY == 0:
                s = S_FINAL * (1 - (1 - step / T) ** 3); masks = masks_at(model, s, rule)
                if prev is not None:   # a weight pruned at an earlier update that is selected again now has regrown between updates
                    for key, mk in masks.items(): regrown[names[key[0]]] += int(((prev[key] == 0) & (mk == 1)).sum())
                apply_masks(model, masks); prev = masks
            # between mask updates the network trains DENSE: pruned weights receive gradients and may regrow
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
            step += 1
        if verbose: print(f'    schedule epoch {ep}, sparsity target {S_FINAL * (1 - (1 - min(step, T) / T) ** 3):.3f}')
    masks = masks_at(model, S_FINAL, rule)
    for key, mk in masks.items(): regrown[names[key[0]]] += int(((prev[key] == 0) & (mk == 1)).sum())
    apply_masks(model, masks); model.regrown = regrown
    # fixed-mask phase: fresh optimizer so stale Adam momentum from the dense phase cannot move masked weights off zero
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    for ep in range(FINAL_EPOCHS):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step(); apply_masks(model, masks)
        if verbose: print(f'    fixed-mask epoch {ep}')
    for h in hooks: h.remove()
    apply_masks(model, masks)
    return model.eval(), le, scaler

def live_filters(model):
    names = layer_names(model); conv0 = [mod for mod, _ in prunable(model) if names[mod] == 'conv.0'][0]
    nz = (conv0.weight.detach() != 0).reshape(conv0.weight.shape[0], -1); return int(nz.any(dim=1).sum()), int(nz.sum())

m_chk, _, _, _ = load_anchor(DATASET, ARCH, BASE_CELL, ANCHOR, arch_kwargs=ARCH_KW)
assert [layer_names(m_chk)[mod] for mod, _ in prunable(m_chk)] == ['conv.0', 'conv.3', 'head']
print('gradual pruner ready')

In [ ]:
# Run both rules on five baselines, with resume
baseline_val, baseline_test, comp_test, macro, rows_ls = {}, {}, {}, [], []
for seed in SEEDS:
    print(f'\n===== seed {seed} =====')
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, BASE_CELL, seed, arch_kwargs=ARCH_KW)
    yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val'); yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
    baseline_val[seed] = per_class_recall_table(yv, pv, le).set_index('label')['recall']; baseline_test[seed] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
    macro.append({'seed': seed, 'cell': 'M0', 'test_macro_f1': f1_score(yt, pt, average='macro')})
    for rule in RULES:
        p_c = PATHS.model(DATASET, ARCH, f'{rule}_paired', seed)
        if os.path.exists(p_c):
            mp = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
            mp.load_state_dict(torch.load(p_c, map_location=DEVICE, weights_only=False)['state_dict']); mp.eval(); print(f'  loaded {rule}')
        else:
            mp, _, _ = gradual_prune(m0, seed, rule, verbose=True); save_ckpt(mp, le, scaler, p_c); print(f'  saved {rule} | regrown weights per layer: {mp.regrown}')
            write_json(OUT / f'gradual_regrown_{rule}_seed{seed}.json', mp.regrown)
        live, taps = live_filters(mp); ls = layer_sparsity(mp)
        rows_ls.append({'seed': seed, 'cell': rule, 'live_filters': live, 'conv0_taps': taps, **ls})
        yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
        comp_test[(seed, rule)] = per_class_recall_table(yt, pc, le).set_index('label')['recall']
        macro.append({'seed': seed, 'cell': rule, 'test_macro_f1': f1_score(yt, pc, average='macro')})
print('\nall runs complete')

In [ ]:
# Aggregate + gate (Notebook 34 schema, prefixed gradual_)
tiers = assign_validation_tiers(pd.DataFrame(baseline_val)); rows = []
for (seed, cell), rc in comp_test.items():
    r0 = baseline_test[seed]
    for cls in r0.index.intersection(rc.index):
        loss = float(r0.loc[cls] - rc.loc[cls]); band = float(tiers.loc[cls, 'validation_2sd_band'])
        rows.append({'seed': seed, 'cell': cell, 'class': cls, 'M0_test_recall': float(r0.loc[cls]), 'compressed_test_recall': float(rc.loc[cls]), 'recall_loss': loss,
                     'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})
eff = pd.DataFrame(rows); assert len(eff) > 0, 'no rows: run the pruning cell in this session first'
eff.to_csv(OUT / 'gradual_per_class_effects.csv', index=False)
summ = eff.groupby(['cell', 'class']).agg(mean_recall_loss=('recall_loss', 'mean'), affected_frequency=('material_and_beyond_band', 'mean')).reset_index(); summ.to_csv(OUT / 'gradual_per_class_summary.csv', index=False)
mdf = pd.DataFrame(macro); mdf.to_csv(OUT / 'gradual_macro_f1_wide.csv', index=False); lsdf = pd.DataFrame(rows_ls); lsdf.to_csv(OUT / 'gradual_layer_sparsity.csv', index=False)
m0m = mdf[mdf.cell == 'M0'].test_macro_f1.mean(); ref = pd.read_csv(OUT / 'cnn_policy_comparison.csv')
tab = []
for rule in RULES:
    g = mdf[mdf.cell == rule].test_macro_f1; l = lsdf[lsdf.cell == rule]
    tab.append({'recipe': rule, 'mean_macro_f1': g.mean(), 'sd': g.std(), 'min': g.min(), 'max': g.max(), 'mean_macro_f1_loss': m0m - g.mean(),
                'classes_affected_ge3of5': int((summ[summ.cell == rule].affected_frequency >= 0.6).sum()), 'conv0_sparsity': float(l['conv.0'].mean()),
                'live_filters': float(l.live_filters.mean()), 'conv3_sparsity': float(l['conv.3'].mean()), 'prunable_sparsity': float(l.prunable_sparsity.mean())})
for _, r in ref.iterrows():
    tab.append({'recipe': 'oneshot_' + r.policy, 'mean_macro_f1': r.mean_macro_f1, 'sd': r.sd_macro_f1, 'min': np.nan, 'max': np.nan, 'mean_macro_f1_loss': r.mean_macro_f1_loss,
                'classes_affected_ge3of5': int(r.classes_affected_ge3of5), 'conv0_sparsity': r.conv0_sparsity, 'live_filters': np.nan, 'conv3_sparsity': r.conv3_sparsity, 'prunable_sparsity': r.prunable_sparsity})
tab = pd.DataFrame(tab); tab.to_csv(OUT / 'gradual_comparison.csv', index=False)
print(f'M0 five-seed macro-F1: {m0m:.4f}\n'); print(tab.round(4).to_string(index=False))
pl = tab[tab.recipe == 'gradual_perlayer80'].iloc[0]; gl = tab[tab.recipe == 'gradual_global80'].iloc[0]
verdict = pd.DataFrame([
 {'criterion': 'perlayer_gradual_conv0_sparsity_ge_0.75', 'value': round(float(pl.conv0_sparsity), 3), 'pass': bool(pl.conv0_sparsity >= 0.75)},
 {'criterion': 'perlayer_gradual_collapse_loss_gt_0.15_and_ge4_classes', 'value': f'loss {pl.mean_macro_f1_loss:.3f}, {int(pl.classes_affected_ge3of5)} classes', 'pass': bool(pl.mean_macro_f1_loss > 0.15 and pl.classes_affected_ge3of5 >= 4)},
 {'criterion': 'global_gradual_keeps_conv0_lt_0.5_sparsity', 'value': round(float(gl.conv0_sparsity), 3), 'pass': bool(gl.conv0_sparsity < 0.5)},
 {'criterion': 'global_gradual_loss_lt_0.05', 'value': round(float(gl.mean_macro_f1_loss), 4), 'pass': bool(gl.mean_macro_f1_loss < 0.05)},
])
print(); print(verdict.to_string(index=False))
print('\nInput-layer mechanism holds under gradual pruning with recovery:', bool(verdict['pass'].all()))
if not bool(verdict['pass'].iloc[1]): print('Per-layer gradual pruning does NOT reproduce the collapse: recovery repairs the input layer; the manuscript scopes its claim to one-shot pruning.')
verdict.to_csv(OUT / 'gradual_gate_verdict.csv', index=False)
write_json(OUT / 'gradual_environment.json', {'rules': RULES, 's_final': S_FINAL, 'sched_epochs': SCHED_EPOCHS, 'final_epochs': FINAL_EPOCHS, 'mask_every': MASK_EVERY, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/41_gradual_pruning_with_recovery.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/gradual_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 41: gradual magnitude pruning with recovery (Zhu-Gupta schedule), per-layer vs global, on five CNN baselines; scope gate for the input-layer mechanism'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)